# Cleaning MSA Files 

In [ ]:
# Convert to Fasta
from Bio import AlignIO
AlignIO.convert("PF10613.alignment.seed", "stockholm", "PF10613.fasta", "fasta")

In [ ]:
# Cleaning pipeline

"""
========================  PF10613 / MSA Cleaning Cheatsheet  ========================

GOAL
- Start from Pfam sequences/MSA, remove junk and bias, keep informative sites.
- Outputs:
    1) <name>.clean.fasta      # after seq-level filtering + Non-Redundant at 90 % sequence identity
    2) <name>.clean.trim.fasta # after column trimming (use this for stats/coev)

THRESHOLDS (what they mean, when to change)
-------------------------------------------------------------------------------------
1) min_frac_len = 0.60   # Fragment filter (seq-level)
   - Keep sequences whose UNGAPPED length >= 0.60 × modal ungapped length.
   - Why: drop partial hits; keep natural indels.
   - Tighten (0.70–0.80): you want only near full-length domains.
   - Loosen (0.50): many fragments/metagenomic sequences.

2) max_frac_amb = 0.05   # Ambiguity filter (seq-level)
   - Max allowed fraction of ambiguous AAs (B/Z/X/J/etc.), ignoring gaps.
   - Why: avoid noise in counts, entropy, matrices.
   - Tighten (0.00–0.03): curated/structural datasets (no ambiguity expected).
   - Loosen (0.10–0.20): very divergent/low-quality data.

3) pid_threshold = 0.90  # Non-redundant filtering (seq-level)
   - Greedy NR: drop sequences with pairwise identity >= 0.90 to any kept rep.
   - Why: reduce over-representation bias; better Neff.
   - Lower (0.70–0.80): more diversity for coevolution.
   - Higher (0.95): keep closely related paralogs/isoforms for motif work.

4) max_gap = 0.20        # Column gap filter (column-level)
   - Keep columns with gap fraction <= 0.20.
   - Why: remove clade-specific insertions/low info positions.
   - Loosen for diverse MSAs (0.30–0.40). Tighten for curated seeds (0.10–0.20).

5) min_consensus = 0.60  # Column consensus filter (column-level)
   - Keep columns where the majority (ignoring gaps) is >= 60%.
   - Why: retain columns with a clear dominant residue (less noise).
   - Raise (0.70–0.80) for conserved motif analysis.
   - Lower (0.40–0.50) for exploratory/co-evolution analyses.

TYPICAL PROFILES (pick one and tweak)
-------------------------------------------------------------------------------------
- "Conservative / publication-ready":
    min_frac_len=0.70, max_frac_amb=0.03, pid_threshold=0.90, max_gap=0.20, min_consensus=0.70
- "Balanced (default)":
    min_frac_len=0.60, max_frac_amb=0.05, pid_threshold=0.90, max_gap=0.20, min_consensus=0.60
- "Exploratory / coevolution-friendly":
    min_frac_len=0.55, max_frac_amb=0.10, pid_threshold=0.80, max_gap=0.35, min_consensus=0.50


=====================================================================================
"""

# =============================================================================
#                    PF10613 MSA Cleaning Script (Biopython)
# =============================================================================
# Threshold constants — edit these values as needed
MIN_FRAC_LEN   = 0.60   # Fragment filter (seq >= 60% of modal ungapped length)
MAX_FRAC_AMB   = 0.05   # Ambiguity filter (<=5% ambiguous AAs: B,Z,X,J,etc)
PID_THRESHOLD  = 0.90   # Non-redundant cutoff (no pairs ≥90% identical)
MAX_GAP        = 0.40   # Column gap filter (≤20% gaps kept)
MIN_CONSENSUS  = 0.40   # Column consensus filter (≥60% majority residue)
# =============================================================================

from Bio import AlignIO
from Bio.Align import MultipleSeqAlignment
from Bio.SeqRecord import SeqRecord
import numpy as np, os

# -------- 0) Helper to auto-detect alignment format --------
def read_alignment_any(path):
    """Attempt to read alignment from .aln/.fasta/.seed with fallback formats."""
    if path.endswith((".seed", ".sto", ".stockholm")):
        formats = ["stockholm"]
    elif path.endswith(".fasta") or path.endswith(".fa"):
        formats = ["fasta"]
    elif path.endswith(".aln"):
        formats = ["clustal", "fasta"]
    else:
        formats = ["clustal", "fasta", "stockholm"]

    tried = []
    for fmt in formats:
        try:
            return AlignIO.read(path, fmt)
        except Exception as e:
            tried.append((fmt, str(e)))
    msg = "Could not read alignment. Tried formats:\n" + "\n".join([f"- {f}: {err}" for f, err in tried])
    raise RuntimeError(msg)

# -------- 1) Sequence-level filters --------
AA_Index = set("ACDEFGHIKLMNPQRSTVWY")

def ungapped_length(seq):
    """Count residues excluding '-' gaps."""
    return sum(ch != '-' for ch in str(seq))

def frac_ambiguous(seq):
    """Fraction of ambiguous AAs (B/Z/X/J etc.) ignoring gaps."""
    s = str(seq).replace("-", "")
    if not s: return 1.0
    amb = sum(ch.upper() not in AA_Index for ch in s)
    return amb / len(s)

def filter_sequences(aln, min_frac_len=MIN_FRAC_LEN, max_frac_amb=MAX_FRAC_AMB):
    """Remove fragments and highly ambiguous sequences."""
    lengths = [ungapped_length(rec.seq) for rec in aln]
    if not lengths: return aln
    modal = max(set(lengths), key=lengths.count)
    keep = [
        rec for rec in aln
        if ungapped_length(rec.seq) >= int(min_frac_len * modal)
        and frac_ambiguous(rec.seq) <= max_frac_amb
    ]
    return MultipleSeqAlignment(keep)

# -------- 2) Non-redundant filtering --------
def pairwise_pid(seqA, seqB):
    """Pairwise percent identity ignoring gaps."""
    A, B = np.array(list(str(seqA))), np.array(list(str(seqB)))
    mask = (A != '-') & (B != '-')
    n = mask.sum()
    return float((A[mask] == B[mask]).mean()) if n else 0.0

def nr_greedy(aln, pid_threshold=PID_THRESHOLD):
    """Keep sequences whose PID < threshold to all representatives."""
    reps = []
    for rec in aln:
        if all(pairwise_pid(rec.seq, r.seq) < pid_threshold for r in reps):
            reps.append(rec)
    return MultipleSeqAlignment(reps)

# -------- 3) Column trimming --------
def trim_columns(aln, max_gap=MAX_GAP, min_consensus=MIN_CONSENSUS):
    """
    Keep columns with gap fraction ≤ max_gap and consensus ≥ min_consensus.
    Returns (trimmed_alignment, kept_mask).
    """
    A = np.array([list(str(r.seq)) for r in aln])
    if A.size == 0: return aln, np.array([], bool)

    gap_frac = (A == '-').mean(0)

    def col_consensus(col):
        col = col[col != '-']
        if col.size == 0: return 0.0
        vals, cnt = np.unique(col, return_counts=True)
        return cnt.max() / cnt.sum()

    cons = np.array([col_consensus(A[:, j]) for j in range(A.shape[1])])
    keep = (gap_frac <= max_gap) & (cons >= min_consensus)
    kept_idx = np.where(keep)[0]

    new_records = []
    for rec in aln:
        arr = np.array(list(str(rec.seq)))
        new_seq = ''.join(arr[kept_idx]) if kept_idx.size else ""
        new_records.append(SeqRecord(rec.seq.__class__(new_seq), id=rec.id, description=""))
    return MultipleSeqAlignment(new_records), keep

# -------- 4) Run pipeline --------
infile = "PF10613.aln" if os.path.exists("PF10613.aln") else "PF10613.fasta"
print(f"Loading alignment from: {infile}")

aln_raw = read_alignment_any(infile)
print(f"Raw: {len(aln_raw)} sequences | {aln_raw.get_alignment_length()} columns")

aln = filter_sequences(aln_raw)
print(f"After length/ambiguity filter: {len(aln)} seq | {aln.get_alignment_length()} cols")

aln_nr = nr_greedy(aln)
print(f"After NR{int(PID_THRESHOLD*100)}: {len(aln_nr)} seq | {aln_nr.get_alignment_length()} cols")

aln_trim, kept_mask = trim_columns(aln_nr)
print(f"After trimming: {len(aln_trim)} seq | {aln_trim.get_alignment_length()} cols")

# -------- 5) Save & QC --------
out_clean = "PF10613.clean.fasta"
out_trim  = "PF10613.clean.trim.fasta"

AlignIO.write(aln_nr, out_clean, "fasta")
AlignIO.write(aln_trim, out_trim, "fasta")

if aln_trim.get_alignment_length() > 0:
    A = np.array([list(str(r.seq)) for r in aln_trim])
    gap_frac_final = (A == '-').mean(0)
    print(f"Mean gap fraction (trimmed): {gap_frac_final.mean():.3f}")
    print(f"Columns with ≤10% gaps: {(gap_frac_final <= 0.10).sum()}")

print("\n[Current Thresholds]")
print(f"MIN_FRAC_LEN  = {MIN_FRAC_LEN}")
print(f"MAX_FRAC_AMB  = {MAX_FRAC_AMB}")
print(f"PID_THRESHOLD = {PID_THRESHOLD}")
print(f"MAX_GAP       = {MAX_GAP}")
print(f"MIN_CONSENSUS = {MIN_CONSENSUS}")

print(f"\nWrote: {out_clean}")
print(f"Wrote: {out_trim}")

In [ ]:
#Inspect what was dropped: 
from Bio import AlignIO
import numpy as np

aln0 = AlignIO.read("PF10613.fasta", "fasta")  

def entropy_per_col(aln):
    A = np.array([list(str(r.seq)) for r in aln])
    ent = []
    for j in range(A.shape[1]):
        col = A[:,j]
        col = col[col!='-']
        if col.size==0: ent.append(np.nan); continue
        vals, cnt = np.unique(col, return_counts=True)
        p = cnt/cnt.sum()
        ent.append(float(-(p*np.log2(p)).sum()))
    return np.array(ent)

# Rebuild kept mask using your current thresholds
MAX_GAP, MIN_CONSENSUS = 0.4, 0.4
A = np.array([list(str(r.seq)) for r in aln0])
gap = (A=='-').mean(0)
def cons(col):
    col = col[col!='-']; 
    if col.size==0: return 0.0
    v,c = np.unique(col, return_counts=True); 
    return c.max()/c.sum()
consensus = np.array([cons(A[:,j]) for j in range(A.shape[1])])
keep = (gap<=MAX_GAP) & (consensus>=MIN_CONSENSUS)

ent = entropy_per_col(aln0)
print("Median entropy (all):", np.nanmedian(ent))
print("Median entropy (kept):   ", np.nanmedian(ent[keep]))
print("Median entropy (dropped):", np.nanmedian(ent[~keep]))

In [1]:
def henikoff_weights(aln):
    A = np.array([list(str(r.seq)) for r in aln])
    nseq, ncol = A.shape
    w = np.zeros(nseq)
    for j in range(ncol):
        col = A[:,j]; mask = col!='-'
        syms, cnt = np.unique(col[mask], return_counts=True)
        if cnt.size==0: continue
        k = cnt.size; look = dict(zip(syms, cnt))
        for i in range(nseq):
            if mask[i]: w[i] += 1.0/(k*look[col[i]])
    w /= w.sum()
    return w

aln = AlignIO.read("PF10613.clean.trim.fasta","fasta")
W = henikoff_weights(aln)
Neff = 1.0 / (W**2).sum()
L = aln.get_alignment_length()
print(f"Neff={Neff:.1f}, L={L}, Neff/L={Neff/L:.2f}")

NameError: name 'AlignIO' is not defined

In [2]:
# MSA Visulaization
from collections import Counter
from IPython.display import HTML, display
import html

# --- Load your alignment ---
aln = AlignIO.read("PF10613.clean.trim.fasta", "fasta")  # change name/format if needed

# --- Helpers ---
KD = dict(A=1.8,R=-4.5,N=-3.5,D=-3.5,C=2.5,Q=-3.5,E=-3.5,G=-0.4,H=-3.2,
          I=4.5,L=3.8,K=-3.9,M=1.9,F=2.8,P=-1.6,S=-0.8,T=-0.7,W=-0.9,Y=-1.3,V=4.2)

def alignment_window(aln, start, end):
    """Return list[(id, subseq)] for 1-based inclusive window [start, end]."""
    L = aln.get_alignment_length()
    start = max(1, start); end = min(L, end)
    s0, e0 = start-1, end
    return [(rec.id, str(rec.seq[s0:e0])) for rec in aln]

def consensus_of(seqs):
    """Ungapped majority consensus over equal-length strings."""
    L = len(seqs[0])
    cons = []
    for i in range(L):
        col = [s[i] for s in seqs if s[i] != '-']
        cons.append(Counter(col).most_common(1)[0][0] if col else '-')
    return "".join(cons)

def pid_to_cons(seq, cons):
    m = sum(1 for a,b in zip(seq,cons) if a!='-' and b!='-')
    return (sum(1 for a,b in zip(seq,cons) if a==b and a!='-' and b!='-')/m) if m else 0.0

def mean_kd(seq):
    vals = [KD.get(a, 0.0) for a in seq if a != '-']
    return sum(vals)/len(vals) if vals else 0.0

def color_seq_html(seq, cons):
    """Return HTML with per-residue spans: matches tinted, mismatches accented, gaps faint."""
    out = []
    for a,b in zip(seq, cons):
        if a == '-':
            out.append('<span class="gap">-</span>')
        elif b == '-' or a != b:
            # mismatch
            out.append(f'<span class="mismatch">{html.escape(a)}</span>')
        else:
            # match to consensus
            out.append(f'<span class="match">{html.escape(a)}</span>')
    return "".join(out)

def render_window_table(aln, start, end, top_n=15, sort_by="pid"):
    rows = alignment_window(aln, start, end)
    ids, seqs = zip(*rows)
    cons = consensus_of(seqs)

    stats = []
    for rec_id, s in rows:
        stats.append({
            "id": rec_id,
            "seq_html": color_seq_html(s, cons),
            "pid": pid_to_cons(s, cons),
            "kd":  mean_kd(s)
        })
    # sort by %id (default) or hydropathy
    stats.sort(key=(lambda r: (-r["pid"], -r["kd"])) if sort_by=="pid" else (lambda r: (-r["kd"], -r["pid"])))
    stats = stats[:top_n]

    # Build HTML -----THIS IS SOOOOOO COOL! I see why everyone loves Jupyter Notebook now! -------
    width = end - start + 1
    ruler = "".join(str((i+1)%10) for i in range(width))

    css = """
    <style>
    .msa-table {border-collapse: collapse; font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace; font-size: 12.5px;}
    .msa-table th, .msa-table td {border-bottom: 1px solid #e7e7e7; padding: 6px 8px; vertical-align: top;}
    .msa-table th {background: #f6f8fa; text-align: left;}
    .seq {letter-spacing: 0.5px; white-space: nowrap;}
    .cons {background:#fffbe6;}
    .ruler {color:#666;}
    .match {background: #e7f7ee;}      /* soft green for matches */
    .mismatch {background: #ffecec;}   /* soft red for mismatches */
    .gap {color:#bbb;}
    .meta {color:#333; font-variant-numeric: tabular-nums;}
    .tag {display:inline-block; background:#eef; color:#224; padding:2px 6px; border-radius:6px; margin-left:6px; font-size:11px;}
    </style>
    """

    header = f"""
    <div class="meta"><strong>Window {start}-{end}</strong>
      <span class="tag">length {width}</span>
      <span class="tag">top {len(stats)} seqs</span>
    </div>
    <table class="msa-table">
      <thead>
        <tr><th>Sequence ID</th><th>Fragment</th><th>%ID to cons</th><th>Mean KD</th></tr>
      </thead>
      <tbody>
        <tr>
          <td class="ruler">ruler</td>
          <td class="seq ruler">{ruler}</td>
          <td class="ruler"></td>
          <td class="ruler"></td>
        </tr>
        <tr>
          <td class="cons">consensus</td>
          <td class="seq cons">{color_seq_html(cons, cons)}</td>
          <td class="cons"></td>
          <td class="cons"></td>
        </tr>
    """

    rows_html = "\n".join(
        f'<tr><td>{html.escape(r["id"])}</td>'
        f'<td class="seq">{r["seq_html"]}</td>'
        f'<td class="meta">{r["pid"]*100:5.1f}%</td>'
        f'<td class="meta">{r["kd"]:5.2f}</td></tr>'
        for r in stats
    )

    footer = "</tbody></table>"
    return HTML(css + header + rows_html + footer)

# --- Render a few windows ---
display(render_window_table(aln, 1, 15, top_n=12)) # First slice
display(render_window_table(aln, 16, 30, top_n=12))
display(render_window_table(aln, 31, 45, top_n=12))
display(render_window_table(aln, 100, 115, top_n=12))

NameError: name 'AlignIO' is not defined